In [2]:
import os
import pandas as pd

all_ratings = pd.read_csv('Ch4_MoviePrediction_AffinityAlalysis_u.data', delimiter="\t",
header=None, names = ["UserID", "MovieID", "Rating", "Datetime"])
all_ratings["Datetime"] = pd.to_datetime(all_ratings['Datetime'],
unit='s')
all_ratings[:5]

,UserID,MovieID,Rating,Datetime
0,196,242,3,1997-12-04 15:55:49
1,186,302,3,1998-04-04 19:22:22
2,22,377,1,1997-11-07 07:18:36
3,244,51,2,1997-11-27 05:02:03
4,166,346,1,1998-02-02 05:33:16


In [3]:
all_ratings["Favorable"] = all_ratings["Rating"] > 3
all_ratings[10:15]

,UserID,MovieID,Rating,Datetime,Favorable
10,62,257,2,1997-11-12 22:07:14,False
11,286,1014,5,1997-11-17 15:38:45,True
12,200,222,5,1997-10-05 09:05:40,True
13,210,40,3,1998-03-27 21:59:54,False
14,224,29,3,1998-02-21 23:40:57,False


In [4]:
ratings = all_ratings[all_ratings['UserID'].isin(range(200))]
favorable_ratings = ratings[ratings["Favorable"]]
favorable_reviews_by_users = dict((k, frozenset(v.values)) 
                                  for k, v in favorable_ratings.groupby("UserID")["MovieID"])

num_favorable_by_movie = ratings[["MovieID", "Favorable"]].groupby("MovieID").sum()
num_favorable_by_movie.sort_values("Favorable", ascending=False)[:5]


,Favorable
MovieID,
50,100
100,89
258,83
181,79
174,74


In [6]:
print (all_ratings[all_ratings['UserID']==675].sort_values("MovieID", ascending=True))

       UserID  MovieID  Rating            Datetime  Favorable
81098     675       86       4 1998-03-10 00:26:14       True
90696     675      223       1 1998-03-10 00:35:51      False
92650     675      235       1 1998-03-10 00:35:51      False
95459     675      242       4 1998-03-10 00:08:42       True
82845     675      244       3 1998-03-10 00:29:35      False
53293     675      258       3 1998-03-10 00:11:19      False
97286     675      269       5 1998-03-10 00:08:07       True
93720     675      272       3 1998-03-10 00:07:11      False
73389     675      286       4 1998-03-10 00:07:11       True
77524     675      303       5 1998-03-10 00:08:42       True
47367     675      305       4 1998-03-10 00:09:08       True
44300     675      306       5 1998-03-10 00:08:07       True
53730     675      311       3 1998-03-10 00:10:47      False
54284     675      312       2 1998-03-10 00:10:24      False
63291     675      318       5 1998-03-10 00:21:13       True
87082   

In [16]:
user_id = 675
user_ratings = all_ratings[all_ratings['UserID'] == user_id]

print(f"Total ratings (rows) by user {user_id}: {len(user_ratings)}")
print(f"Unique movies rated by user {user_id}: {user_ratings['MovieID'].nunique()}")

Total ratings (rows) by user 675: 34
Unique movies rated by user 675: 34


In [15]:
print (all_ratings[all_ratings['UserID']==675].value_counts("Favorable"))
print (all_ratings[all_ratings['UserID']==675].value_counts("Rating"))

Favorable
True     23
False    11
Name: count, dtype: int64
Rating
5    12
4    11
1     4
3     4
2     3
Name: count, dtype: int64


In [23]:
print(f"Total unique users: {all_ratings['UserID'].nunique()}")


Total unique users: 943


In [40]:
user_id = 405
user_ratings = all_ratings[all_ratings['UserID'] == user_id]

print(f"Total ratings (rows) by user {user_id}: {len(user_ratings)}")
print(f"Unique movies rated by user {user_id}: {user_ratings['MovieID'].nunique()}")

# counts of ratings per user (descending)
all_ratings['UserID'].value_counts().head(100)

Total ratings (rows) by user 405: 737
Unique movies rated by user 405: 737


UserID
405    737
655    685
13     636
450    540
276    518
      ... 
506    242
932    241
886    240
798    239
244    238
Name: count, Length: 100, dtype: int64

In [26]:
user_counts = all_ratings.groupby('UserID').size().reset_index(name='num_ratings')
user_counts.sort_values('num_ratings', ascending=False).head(10)

,UserID,num_ratings
404,405,737
654,655,685
12,13,636
449,450,540
275,276,518
415,416,493
536,537,490
302,303,484
233,234,480
392,393,448


In [27]:
user_stats = all_ratings.groupby('UserID').agg(
    num_ratings=('Rating','size'),
    avg_rating=('Rating','mean')
).sort_values('num_ratings', ascending=False)
user_stats.head(10)

,num_ratings,avg_rating
UserID,,
405,737,1.834464
655,685,2.908029
13,636,3.097484
450,540,3.864815
276,518,3.465251
416,493,3.845842
537,490,2.865306
303,484,3.365702
234,480,3.122917


In [29]:
# Ratings distribution per user
pd.crosstab(all_ratings['UserID'], all_ratings['Rating']).head(10)

Rating,1,2,3,4,5
UserID,,,,,
1,25,28,56,82,81
2,4,1,17,27,13
3,8,16,15,9,6
4,0,1,4,5,14
5,42,22,53,32,26
6,7,27,43,93,41
7,14,18,97,113,161
8,5,4,10,19,21
9,1,0,1,10,10


In [ ]:
import sys

frequent_itemsets = {}
min_support = 50
frequent_itemsets[1] = dict((frozenset((movie_id,)),
row["Favorable"])
for movie_id, row in num_favorable_by_movie.iterrows()
if row["Favorable"] > min_support)

from collections import defaultdict
def find_frequent_itemsets(favorable_reviews_by_users, k_1_itemsets,
min_support):
    counts = defaultdict(int)
    for user, reviews in favorable_reviews_by_users.items():
        for itemset in k_1_itemsets:
            if itemset.issubset(reviews):
                for other_reviewed_movie in reviews - itemset:
                    larger_itemset = itemset | frozenset((other_reviewed_movie,))
                    counts[larger_itemset] += 1
    return dict([(itemset, frequency) for itemset, frequency in
counts.items() if frequency >= min_support])
for k in range(2, 20):
    cur_frequent_itemsets = find_frequent_itemsets(favorable_reviews_by_users,
                            frequent_itemsets[k-1], min_support)
    frequent_itemsets[k] = cur_frequent_itemsets
    if len(cur_frequent_itemsets) == 0:
        print ("Did not find any frequent itemsets of length {}".format(k))
        sys.stdout.flush()
        break
    else:
        print("I found {} frequent itemsets of length{}".format(len(cur_frequent_itemsets), k))
        sys.stdout.flush()
del frequent_itemsets[1]

I found 93 frequent itemsets of length2
I found 295 frequent itemsets of length3
I found 593 frequent itemsets of length4
I found 785 frequent itemsets of length5
I found 677 frequent itemsets of length6
I found 373 frequent itemsets of length7
I found 126 frequent itemsets of length8
I found 24 frequent itemsets of length9
I found 2 frequent itemsets of length10
Did not find any frequent itemsets of length 11


In [ ]:
candidate_rules = []
for itemset_length, itemset_counts in frequent_itemsets.items():
    #new condition
    #if itemset_length < 2:
        #continue

    for itemset in itemset_counts.keys():
        for conclusion in itemset:
            premise = itemset - set((conclusion,))
            candidate_rules.append((premise, conclusion))
print(candidate_rules[:5])

[(frozenset({np.int64(7)}), 1), (frozenset({1}), np.int64(7)), (frozenset({np.int64(50)}), 1), (frozenset({1}), np.int64(50)), (frozenset({1}), np.int64(56))]


In [ ]:
#debugging

for k, v in frequent_itemsets.items():
    print("Itemset length:", k)
    for item in v.keys():
        print(item)

Itemset length: 2
frozenset({1, np.int64(7)})
frozenset({1, np.int64(50)})
frozenset({np.int64(56), 1})
frozenset({np.int64(64), 1})
frozenset({1, np.int64(79)})
frozenset({1, np.int64(98)})
frozenset({1, np.int64(100)})
frozenset({1, np.int64(127)})
frozenset({1, np.int64(172)})
frozenset({1, np.int64(174)})
frozenset({1, np.int64(181)})
frozenset({np.int64(9), 7})
frozenset({np.int64(50), 7})
frozenset({np.int64(56), 7})
frozenset({np.int64(64), 7})
frozenset({np.int64(79), 7})
frozenset({np.int64(98), 7})
frozenset({np.int64(100), 7})
frozenset({np.int64(127), 7})
frozenset({np.int64(172), 7})
frozenset({np.int64(174), 7})
frozenset({np.int64(181), 7})
frozenset({np.int64(258), 7})
frozenset({9, np.int64(50)})
frozenset({np.int64(56), 9})
frozenset({np.int64(64), 9})
frozenset({9, np.int64(98)})
frozenset({9, np.int64(100)})
frozenset({9, np.int64(127)})
frozenset({9, np.int64(174)})
frozenset({9, np.int64(181)})
frozenset({np.int64(56), 50})
frozenset({np.int64(64), 50})
frozenset(

In [ ]:
import pandas as pd

df = pd.read_csv("Ch4_MoviePrediction_AffinityAlalysis_u.info", sep="|", header=None, encoding="latin-1")
print(df.head())

                0
0       943 users
1      1682 items
2  100000 ratings
